

###  Objective

This notebook serves as the **backend prediction engine** for the Scam Detection Frontend. It accepts raw text from four input sources:

1.  Job / Internship Descriptions
2.  Scraped URL Content
3.  OCR-Extracted Screenshot Text
4.  PDF Offer Letter Text

All inputs are normalized to plain text and passed through the same hybrid engine.

###  Output Per Analysis
- `risk_score` — Numeric risk (0–100)
- `risk_level` — LOW / MEDIUM / HIGH
- `trust_score` — Numeric trust (0–100)
- `trust_level` — High Trust / Moderate Trust / Low Trust
- `ml_probability` — Raw ML model probability
- `fraud_reasons` — Human-readable explanation list
- `recommended_action` — Actionable guidance
- Full JSON response


In [1]:
import pandas as pd   
import numpy as np         
import joblib                
import re                    
import json                  
import textstat              
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully.")
print(f"   pandas   : {pd.__version__}")
print(f"   numpy    : {np.__version__}")
print(f"   joblib   : {joblib.__version__}")
print(f"   textstat : {textstat.__version__}")

Libraries imported successfully.
   pandas   : 2.2.2
   numpy    : 1.26.4
   joblib   : 1.4.2
   textstat : (0, 7, 13)


In [2]:
#  STEP 2 : Load Saved Model

MODEL_PATH        = "../models/best_scam_detector.pkl"
FEATURE_NAMES_PATH = "../models/feature_names.pkl"

# Load the trained classifier
model = joblib.load(MODEL_PATH)
print(f"Model loaded   : {MODEL_PATH}")
print(f"   Model type     : {type(model).__name__}")

# Load the ordered feature name list
feature_names = joblib.load(FEATURE_NAMES_PATH)
print(f"\n Feature names loaded : {FEATURE_NAMES_PATH}")
print(f"   Total features       : {len(feature_names)}")
print(f"   Feature preview      : {feature_names[:10]} ...")

Model loaded   : ../models/best_scam_detector.pkl
   Model type     : XGBClassifier

 Feature names loaded : ../models/feature_names.pkl
   Total features       : 29
   Feature preview      : ['cleaned_stipend_monthly', 'has_website', 'has_email', 'skills_count', 'title_word_count', 'desc_char_length', 'numeric_openings', 'is_bulk_hiring', 'is_wfh_short_duration', 'company_name_word_count'] ...


In [3]:
#  STEP 3 : Load Fraud Dictionaries

# --- 3A. Monetary / Fee-Based Fraud Phrases ---
fraud_phrases = [
    "registration fee", "security deposit", "processing fee",
    "training fee", "joining fee", "application fee",
    "background check fee", "kit fee", "equipment fee",
    "investment required", "earn money fast", "make money online",
    "guaranteed income", "no experience needed", "work from home earn",
    "money transfer", "wire transfer", "pay upfront",
    "advance payment", "refundable deposit", "non-refundable",
    "pay to work", "data entry work", "typing work from home",
    "part time earn", "unlimited earnings", "get rich",
    "lottery winner", "inheritance", "100% profit"
]

# --- 3B. Urgency / Pressure Language ---
urgency_words = [
    "apply now", "limited seats", "hurry", "urgent hiring",
    "last chance", "immediate joining", "only today",
    "few slots left", "don't miss", "act now",
    "respond immediately", "fast response required",
    "deadline today", "interview today", "join today",
    "selected candidates only", "exclusive offer",
    "once in a lifetime", "don't wait", "instant selection"
]

# --- 3C. Risky / Anonymous Contact Channels ---
risky_contact_terms = [
    "telegram", "whatsapp", "gmail.com", "yahoo.com",
    "hotmail.com", "rediffmail.com", "outlook.com",
    "call now", "contact on whatsapp", "message on telegram",
    "wechat", "signal", "contact via sms",
    "no official website", "personal number",
    "direct contact", "private recruiter"
]

print(" Fraud dictionaries loaded.")
print(f"   fraud_phrases       : {len(fraud_phrases)} entries")
print(f"   urgency_words       : {len(urgency_words)} entries")
print(f"   risky_contact_terms : {len(risky_contact_terms)} entries")

 Fraud dictionaries loaded.
   fraud_phrases       : 30 entries
   urgency_words       : 20 entries
   risky_contact_terms : 17 entries


---
## STEP 4 : Build NLP Feature Generator

This function replicates the NLP feature generation logic from **Notebook 3A**.
It computes readability and linguistic structure metrics from raw text.

Features generated:
| Feature | Description |
|---|---|
| `readability_score` | Flesch Reading Ease score (0–100, higher = easier) |
| `word_count` | Total number of words |
| `sentence_count` | Total number of sentences |
| `average_word_length` | Mean character length per word |
| `unique_word_count` | Count of unique vocabulary tokens |
| `lexical_diversity` | Ratio of unique words to total words |

In [4]:
#  STEP 4 : NLP Feature Generator

def generate_nlp_features(text: str) -> dict:
    """
    Compute NLP-based features from a raw text string.

    Parameters
    ----------
    text : str
        Raw input text (job description, URL content, OCR output, etc.)

    Returns
    -------
    dict
        Dictionary of NLP feature names → values.
    """
    if not text or not isinstance(text, str):
        text = ""

    text_clean = text.strip()

    # Tokenise words (alphanumeric only)
    words = re.findall(r'\b[a-zA-Z]+\b', text_clean.lower())

    # Sentence count via punctuation heuristic
    sentences = re.split(r'[.!?]+', text_clean)
    sentences = [s.strip() for s in sentences if s.strip()]

    word_count      = len(words)
    sentence_count  = max(len(sentences), 1)
    unique_words    = set(words)
    unique_word_count = len(unique_words)

    avg_word_length = (
        round(sum(len(w) for w in words) / word_count, 4)
        if word_count > 0 else 0
    )

    lexical_diversity = (
        round(unique_word_count / word_count, 4)
        if word_count > 0 else 0
    )

    # Flesch Reading Ease (textstat)
    try:
        readability_score = textstat.flesch_reading_ease(text_clean)
    except Exception:
        readability_score = 50.0   # neutral default

    return {
        "readability_score"  : readability_score,
        "word_count"         : word_count,
        "sentence_count"     : sentence_count,
        "average_word_length": avg_word_length,
        "unique_word_count"  : unique_word_count,
        "lexical_diversity"  : lexical_diversity,
    }


# ── Quick sanity check ──
sample = "We are hiring immediately! Send registration fee of Rs.500 to our WhatsApp number."
_nlp = generate_nlp_features(sample)
print("✅ NLP Feature Generator ready.")
print("   Sample output:", json.dumps(_nlp, indent=6))

✅ NLP Feature Generator ready.
   Sample output: {
      "readability_score": 46.2520512820513,
      "word_count": 13,
      "sentence_count": 3,
      "average_word_length": 4.9231,
      "unique_word_count": 13,
      "lexical_diversity": 1.0
}


---
## STEP 5 : Build Feature Engineering Generator

This function replicates the structured feature engineering logic from **Notebook 3**.
It computes keyword and pattern-density features that capture fraud signals.

Features generated:
| Feature | Description |
|---|---|
| `keyword_count` | Total fraud / urgency / contact keyword hits |
| `fraud_phrase_score` | Count of matched fraud_phrases |
| `urgency_score` | Count of matched urgency_words |
| `contact_risk_score` | Count of matched risky_contact_terms |
| `all_caps_ratio` | Proportion of words written in ALL CAPS |
| `text_length` | Total character length of the input |

In [5]:
#  STEP 5 : Feature Engineering Generator

def generate_engineered_features(text: str) -> dict:
    """
    Compute keyword density and structural fraud features.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    dict
        Dictionary of engineered feature names → values.
    """
    if not text or not isinstance(text, str):
        text = ""

    text_lower  = text.lower()
    text_length = len(text)

    # Count fraud phrase matches
    fraud_phrase_score = sum(
        1 for phrase in fraud_phrases if phrase in text_lower
    )

    # Count urgency word matches
    urgency_score = sum(
        1 for word in urgency_words if word in text_lower
    )

    # Count risky contact term matches
    contact_risk_score = sum(
        1 for term in risky_contact_terms if term in text_lower
    )

    # Total keyword hits
    keyword_count = fraud_phrase_score + urgency_score + contact_risk_score

    # ALL-CAPS word ratio — scams frequently shout at the reader
    all_words  = text.split()
    caps_words = [w for w in all_words if w.isupper() and len(w) > 1]
    all_caps_ratio = (
        round(len(caps_words) / len(all_words), 4)
        if all_words else 0
    )

    return {
        "keyword_count"      : keyword_count,
        "fraud_phrase_score" : fraud_phrase_score,
        "urgency_score"      : urgency_score,
        "contact_risk_score" : contact_risk_score,
        "all_caps_ratio"     : all_caps_ratio,
        "text_length"        : text_length,
    }


# ── Quick sanity check ──
_eng = generate_engineered_features(sample)
print(" Feature Engineering Generator ready.")
print("   Sample output:", json.dumps(_eng, indent=6))

 Feature Engineering Generator ready.
   Sample output: {
      "keyword_count": 2,
      "fraud_phrase_score": 1,
      "urgency_score": 0,
      "contact_risk_score": 1,
      "all_caps_ratio": 0.0,
      "text_length": 82
}


---
## STEP 6 : Create Model Input Generator

The trained model expects features in a **specific column order** defined by `feature_names.pkl`.

This function:
1. Merges NLP features + Engineered features into a single flat dictionary
2. Reorders them to exactly match `feature_names`
3. Fills any missing features with `0` (safe default)
4. Returns a `(1, n_features)` NumPy array ready for `model.predict_proba()`

In [6]:
#  STEP 6 : Model Input Generator

def build_model_input(text: str) -> np.ndarray:
    """
    Convert raw text into a feature array aligned with training.

    Parameters
    ----------
    text : str
        Raw input text from any source.

    Returns
    -------
    np.ndarray, shape (1, n_features)
        Feature vector ready for model.predict / predict_proba.
    """
    # Gather all computed features
    nlp_feats = generate_nlp_features(text)
    eng_feats = generate_engineered_features(text)

    all_features = {**nlp_feats, **eng_feats}

    # Align to training feature order — missing keys → 0
    ordered_values = [
        all_features.get(fname, 0) for fname in feature_names
    ]

    return np.array(ordered_values).reshape(1, -1)


# ── Quick sanity check ──
_arr = build_model_input(sample)
print("Model Input Generator ready.")
print(f"   Output shape : {_arr.shape}")
print(f"   Feature values (first 8): {_arr[0][:8].tolist()}")

✅ Model Input Generator ready.
   Output shape : (1, 29)
   Feature values (first 8): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


---
## STEP 7 : Generate ML Prediction

The pre-trained model generates two outputs:
- `ml_prediction` — binary class label (0 = genuine, 1 = scam)
- `ml_probability` — float probability of the scam class, range `[0, 1]`

The probability is used downstream in the **Hybrid Risk Engine** (Step 9).

In [7]:
#  STEP 7 : ML Prediction Generator


def get_ml_prediction(text: str) -> dict:
    """
    Run ML inference on raw input text.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    dict
        ml_prediction (int) and ml_probability (float).
    """
    X = build_model_input(text)

    ml_prediction   = int(model.predict(X)[0])

    # predict_proba returns [[prob_class_0, prob_class_1]]
    proba_array     = model.predict_proba(X)[0]
    ml_probability  = round(float(proba_array[1]), 4)   # scam probability

    return {
        "ml_prediction" : ml_prediction,
        "ml_probability": ml_probability,
    }


# ── Quick sanity check ──
_ml = get_ml_prediction(sample)
print(" ML Prediction Generator ready.")
print(f"   ml_prediction  : {_ml['ml_prediction']} (1 = scam, 0 = genuine)")
print(f"   ml_probability : {_ml['ml_probability']}")

✅ ML Prediction Generator ready.
   ml_prediction  : 0 (1 = scam, 0 = genuine)
   ml_probability : 0.1588


---
## STEP 8 : Rule-Based Fraud Engine

A deterministic, weighted scoring engine that assigns a **rule_score** (0–100) based on how many fraud signals are detected.

This engine is **complementary** to the ML model. It catches patterns that statistical models sometimes miss (e.g., brand-new phrasing) and provides **explainability**.

### Scoring Weights
| Signal | Weight |
|--------|--------|
| Fraud phrase (each) | +10 |
| Urgency word (each) | +6 |
| Risky contact term (each) | +8 |
| Anonymous recruiter (no company name / website) | +15 |
| Excessive ALL-CAPS (ratio > 0.15) | +5 |
| Very short posting (< 50 words) | +10 |
| Suspicious phone / email pattern | +7 |

In [8]:
#  STEP 8 : Rule-Based Fraud Engine

# Pattern detectors
_PHONE_PATTERN = re.compile(r'(\+?\d[\d\s\-]{8,}\d)')
_PERSONAL_EMAIL_PATTERN = re.compile(
    r'\b[\w.+-]+@(gmail|yahoo|hotmail|rediffmail|outlook|ymail)\.com\b',
    re.IGNORECASE
)
_NO_WEBSITE_INDICATORS = [
    "no website", "visit us at", "contact directly",
    "no official site", "personal recruiter"
]
_ANON_RECRUITER_INDICATORS = [
    "company name not disclosed", "client of ours",
    "undisclosed company", "anonymous client",
    "confidential company", "our client"
]


def compute_rule_score(text: str) -> dict:
    """
    Compute a weighted rule-based fraud score.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    dict
        rule_score (int, 0–100) and raw_rule_points (int, pre-cap).
    """
    if not text:
        return {"rule_score": 0, "raw_rule_points": 0}

    text_lower  = text.lower()
    raw_points  = 0

    # ── Signal 1 : Fraud phrases (+10 each) ──
    fp_hits = sum(1 for p in fraud_phrases if p in text_lower)
    raw_points += fp_hits * 10

    # ── Signal 2 : Urgency words (+6 each) ──
    uw_hits = sum(1 for w in urgency_words if w in text_lower)
    raw_points += uw_hits * 6

    # ── Signal 3 : Risky contact terms (+8 each) ──
    rc_hits = sum(1 for t in risky_contact_terms if t in text_lower)
    raw_points += rc_hits * 8

    # ── Signal 4 : Anonymous recruiter indicators (+15) ──
    if any(ind in text_lower for ind in _ANON_RECRUITER_INDICATORS):
        raw_points += 15

    # ── Signal 5 : Excessive ALL-CAPS ratio (+5) ──
    words_all  = text.split()
    caps_words = [w for w in words_all if w.isupper() and len(w) > 1]
    caps_ratio = len(caps_words) / len(words_all) if words_all else 0
    if caps_ratio > 0.15:
        raw_points += 5

    # ── Signal 6 : Very short posting — low information (<50 words, +10) ──
    word_count = len(re.findall(r'\b\w+\b', text))
    if word_count < 50:
        raw_points += 10

    # ── Signal 7 : Personal email address detected (+7) ──
    if _PERSONAL_EMAIL_PATTERN.search(text):
        raw_points += 7

    # Cap at 100
    rule_score = min(raw_points, 100)

    return {
        "rule_score"     : rule_score,
        "raw_rule_points": raw_points,
    }


# ── Quick sanity check ──
_rs = compute_rule_score(sample)
print(" Rule-Based Fraud Engine ready.")
print(f"   rule_score      : {_rs['rule_score']}")
print(f"   raw_rule_points : {_rs['raw_rule_points']}")

✅ Rule-Based Fraud Engine ready.
   rule_score      : 28
   raw_rule_points : 28


---
## STEP 9 : Hybrid Risk Engine

The Hybrid Risk Engine combines ML and rule-based scores for a **final_risk_score**.

### Blending Formula
```
final_risk_score = (0.6 × rule_score) + (0.4 × (ml_probability × 100))
```

**Rationale:**
- The rule-based score contributes **60 %** — it directly encodes domain knowledge and is highly interpretable.
- The ML probability contributes **40 %** — it captures statistical patterns beyond the keyword list.
- The result is capped at **100**.

In [9]:

#  STEP 9 : Hybrid Risk Engine

def compute_hybrid_risk(rule_score: float, ml_probability: float) -> float:
    """
    Compute the blended final risk score.

    Parameters
    ----------
    rule_score     : float  Rule-based score (0–100)
    ml_probability : float  ML scam probability (0–1)

    Returns
    -------
    float
        final_risk_score in [0, 100], rounded to 2 dp.
    """
    raw_score = (
        0.6 * rule_score
        + 0.4 * (ml_probability * 100)
    )
    return round(min(raw_score, 100), 2)


# ── Quick sanity check ──
_hybrid = compute_hybrid_risk(
    rule_score     = _rs["rule_score"],
    ml_probability = _ml["ml_probability"]
)
print(" Hybrid Risk Engine ready.")
print(f"   rule_score      : {_rs['rule_score']}")
print(f"   ml_probability  : {_ml['ml_probability']}")
print(f"   final_risk_score: {_hybrid}")

✅ Hybrid Risk Engine ready.
   rule_score      : 28
   ml_probability  : 0.1588
   final_risk_score: 23.15


---
## STEP 10 : Risk Classification

Converts the numeric `final_risk_score` into a categorical `risk_level`.

| Score Range | Risk Level | Interpretation |
|---|---|---|
| 0 – 39 | 🟢 LOW | Unlikely to be a scam |
| 40 – 69 | 🟡 MEDIUM | Requires review before applying |
| 70 – 100 | 🔴 HIGH | Strong scam indicators present |

In [10]:
#  STEP 10 : Risk Classification

def classify_risk(risk_score: float) -> str:
    """
    Return a categorical risk level for a given numeric risk score.

    Parameters
    ----------
    risk_score : float  Final hybrid risk score (0–100)

    Returns
    -------
    str
        'LOW', 'MEDIUM', or 'HIGH'
    """
    if risk_score < 40:
        return "LOW"
    elif risk_score < 70:
        return "MEDIUM"
    else:
        return "HIGH"


# ── Quick sanity check ──
_risk_level = classify_risk(_hybrid)
print(" Risk Classifier ready.")
print(f"   risk_score : {_hybrid}")
print(f"   risk_level : {_risk_level}")

✅ Risk Classifier ready.
   risk_score : 23.15
   risk_level : LOW


---
## STEP 11 : Trust Classification

The Trust Score is the **inverse** of the Risk Score and communicates safety from the job-seeker's perspective.

```
trust_score = 100 − final_risk_score
```

| Trust Score | Trust Level |
|---|---|
| 61 – 100 | ✅ High Trust |
| 31 – 60 | ⚠️ Moderate Trust |
| 0 – 30 | ❌ Low Trust |

In [11]:
#  STEP 11 : Trust Classification

def compute_trust(risk_score: float) -> dict:
    """
    Derive trust score and human-readable trust level.

    Parameters
    ----------
    risk_score : float  Final hybrid risk score (0–100)

    Returns
    -------
    dict
        trust_score (float) and trust_level (str).
    """
    trust_score = round(100 - risk_score, 2)

    if trust_score > 60:
        trust_level = "High Trust"
    elif trust_score > 30:
        trust_level = "Moderate Trust"
    else:
        trust_level = "Low Trust"

    return {
        "trust_score": trust_score,
        "trust_level": trust_level,
    }


# ── Quick sanity check ──
_trust = compute_trust(_hybrid)
print(" Trust Classifier ready.")
print(f"   trust_score : {_trust['trust_score']}")
print(f"   trust_level : {_trust['trust_level']}")

✅ Trust Classifier ready.
   trust_score : 76.85
   trust_level : High Trust


---
## STEP 12 : Explainability Layer

Converts raw signal detections into **human-readable fraud reason strings**.
These are displayed to the end-user in the frontend to explain *why* a posting was flagged.

Each check maps a detected pattern → a readable explanation:

In [12]:

#  STEP 12 : Explainability Layer

# Specific phrase → human label mapping for top fraud phrases
_FRAUD_PHRASE_LABELS = {
    "registration fee"   : "Registration fee mentioned",
    "security deposit"   : "Security deposit required",
    "processing fee"     : "Processing fee detected",
    "training fee"       : "Training fee required",
    "joining fee"        : "Joining fee mentioned",
    "investment required": "Investment required from applicant",
    "earn money fast"    : "'Earn money fast' language detected",
    "guaranteed income"  : "Guaranteed income promise detected",
    "pay upfront"        : "Upfront payment requested",
    "advance payment"    : "Advance payment mentioned",
    "pay to work"        : "Pay-to-work scheme detected",
    "get rich"           : "Get-rich-quick language detected",
    "100% profit"        : "Unrealistic profit guarantee found",
    "wire transfer"      : "Wire transfer payment requested",
    "data entry work"    : "Data entry scam pattern detected",
    "typing work from home": "Typing-job scam pattern detected",
}

_CONTACT_LABELS = {
    "telegram"           : "Telegram contact used (no official channel)",
    "whatsapp"           : "WhatsApp communication detected",
    "gmail.com"          : "Personal Gmail address used for recruitment",
    "yahoo.com"          : "Personal Yahoo address used for recruitment",
    "hotmail.com"        : "Personal Hotmail address used for recruitment",
    "call now"           : "'Call now' pressure tactic detected",
    "wechat"             : "WeChat contact used (no official channel)",
}

_URGENCY_LABELS = {
    "apply now"          : "High-pressure 'Apply Now' language",
    "urgent hiring"      : "Urgent hiring claim detected",
    "limited seats"      : "False scarcity tactic: 'Limited seats'",
    "immediate joining"  : "Immediate joining demanded",
    "instant selection"  : "Instant selection promise detected",
    "deadline today"     : "Same-day deadline pressure",
}


def generate_fraud_reasons(text: str, risk_score: float) -> list:
    """
    Build a human-readable list of fraud indicators found in the text.

    Parameters
    ----------
    text       : str    Raw input text.
    risk_score : float  Final hybrid risk score.

    Returns
    -------
    list of str
        Explanation strings for the frontend.
    """
    reasons    = []
    text_lower = text.lower()

    # ── Check fraud phrases ──
    for phrase, label in _FRAUD_PHRASE_LABELS.items():
        if phrase in text_lower:
            reasons.append(label)

    # ── Check risky contact terms ──
    for term, label in _CONTACT_LABELS.items():
        if term in text_lower:
            reasons.append(label)

    # ── Check urgency words ──
    for word, label in _URGENCY_LABELS.items():
        if word in text_lower:
            reasons.append(label)

    # ── Anonymous recruiter ──
    if any(ind in text_lower for ind in _ANON_RECRUITER_INDICATORS):
        reasons.append("Anonymous/undisclosed recruiter")

    # ── Short posting ──
    if len(re.findall(r'\b\w+\b', text)) < 50:
        reasons.append("Very short job posting — low information content")

    # ── Personal email ──
    if _PERSONAL_EMAIL_PATTERN.search(text):
        reasons.append("Personal email address used for hiring")

    # ── ML-driven catch-all if model flagged it but no specific reason ──
    if risk_score >= 70 and not reasons:
        reasons.append(
            "ML model detected statistical patterns consistent with scam postings"
        )

    # ── If genuinely clean ──
    if not reasons:
        reasons.append("No significant fraud indicators detected")

    return reasons


# ── Quick sanity check ──
_reasons = generate_fraud_reasons(sample, _hybrid)
print("✅ Explainability Layer ready.")
print("   Detected reasons:")
for r in _reasons:
    print(f"     • {r}")

✅ Explainability Layer ready.
   Detected reasons:
     • Registration fee mentioned
     • WhatsApp communication detected
     • Very short job posting — low information content


---
## STEP 13 : Recommendation Engine

Translates the final risk level into a **single, actionable recommendation** for the job-seeker.

| Risk Level | Recommendation |
|---|---|
| LOW | ✅ Safe to Proceed |
| MEDIUM (score 40–54) | 👁️ Review Before Applying |
| MEDIUM (score 55–69) | 🔍 Manual Verification Required |
| HIGH | 🚨 Potential Scam Detected — Do Not Apply |

In [13]:

#  STEP 13 : Recommendation Engine

def get_recommendation(risk_level: str, risk_score: float) -> str:
    """
    Generate an actionable recommendation string.

    Parameters
    ----------
    risk_level : str    'LOW', 'MEDIUM', or 'HIGH'
    risk_score : float  Numeric risk score (0–100)

    Returns
    -------
    str
        Human-readable recommended action.
    """
    if risk_level == "LOW":
        return "Safe to Proceed"

    elif risk_level == "MEDIUM":
        if risk_score < 55:
            return "Review Before Applying"
        else:
            return "Manual Verification Required"

    else:  # HIGH
        return "Potential Scam Detected — Do Not Apply"


# ── Quick sanity check ──
_rec = get_recommendation(_risk_level, _hybrid)
print(" Recommendation Engine ready.")
print(f"   risk_level          : {_risk_level}")
print(f"   risk_score          : {_hybrid}")
print(f"   recommended_action  : {_rec}")

✅ Recommendation Engine ready.
   risk_level          : LOW
   risk_score          : 23.15
   recommended_action  : Safe to Proceed


---
## STEP 14 : JSON Response Generator — `analyze_job()`

The primary public function of this engine.

It orchestrates all prior steps and returns a **single structured JSON response** for every input text.

### Function Signature
```python
analyze_job(raw_text: str) -> dict
```

### Output Schema
```json
{
  "risk_score"         : 84.5,
  "risk_level"         : "HIGH",
  "trust_score"        : 15.5,
  "trust_level"        : "Low Trust",
  "ml_prediction"      : 1,
  "ml_probability"     : 0.87,
  "rule_score"         : 80,
  "fraud_reasons"      : ["Registration fee mentioned", ...],
  "recommended_action" : "Potential Scam Detected — Do Not Apply"
}
```

In [14]:

#  STEP 14 : JSON Response Generator

def analyze_job(raw_text: str) -> dict:
    """
    Full pipeline: raw text → structured JSON fraud analysis.

    Parameters
    ----------
    raw_text : str
        Any text input — job description, URL content,
        OCR output, or offer letter text.

    Returns
    -------
    dict
        Complete fraud analysis response.
    """
    # ── 1. ML Prediction ──────────────────────────────────────
    ml_result = get_ml_prediction(raw_text)

    # ── 2. Rule-Based Scoring ─────────────────────────────────
    rule_result = compute_rule_score(raw_text)

    # ── 3. Hybrid Risk Score ──────────────────────────────────
    final_risk = compute_hybrid_risk(
        rule_score     = rule_result["rule_score"],
        ml_probability = ml_result["ml_probability"]
    )

    # ── 4. Risk Level ─────────────────────────────────────────
    risk_level = classify_risk(final_risk)

    # ── 5. Trust Score ────────────────────────────────────────
    trust = compute_trust(final_risk)

    # ── 6. Explainability ─────────────────────────────────────
    reasons = generate_fraud_reasons(raw_text, final_risk)

    # ── 7. Recommendation ─────────────────────────────────────
    recommendation = get_recommendation(risk_level, final_risk)

    # ── 8. Assemble Response ──────────────────────────────────
    return {
        "risk_score"        : final_risk,
        "risk_level"        : risk_level,
        "trust_score"       : trust["trust_score"],
        "trust_level"       : trust["trust_level"],
        "ml_prediction"     : ml_result["ml_prediction"],
        "ml_probability"    : ml_result["ml_probability"],
        "rule_score"        : rule_result["rule_score"],
        "fraud_reasons"     : reasons,
        "recommended_action": recommendation,
    }


print("✅ analyze_job() function defined and ready.")
print("   Entry-point for all frontend API calls.")

✅ analyze_job() function defined and ready.
   Entry-point for all frontend API calls.


---
## STEP 15 : Frontend Simulation

Two complete end-to-end tests to verify the engine:

### Test A — Genuine Job Description
A legitimate software engineering position with a named company, clear requirements, and a professional website.

### Test B — Scam Job Description
A fraudulent posting containing multiple red flags: registration fee, WhatsApp contact, urgency pressure, and vague role description.

In [15]:

#  STEP 15 : Frontend Simulation

# ══════════════════════════════════════════════════════════════
#  TEST A : Genuine Job Description
# ══════════════════════════════════════════════════════════════
genuine_job = """
Software Engineer — Backend (Python)
Company: TechNova Solutions Pvt. Ltd.
Website: www.technova.io/careers

About Us:
TechNova Solutions is a Bengaluru-based B2B SaaS company with over 200 employees,
serving clients across fintech, edtech, and logistics verticals.

Responsibilities:
- Design and maintain RESTful APIs using Django REST Framework
- Optimise PostgreSQL queries for high-throughput transactional systems
- Participate in code reviews and architecture discussions
- Collaborate with frontend engineers and product managers

Requirements:
- 2–4 years of Python backend development experience
- Strong knowledge of SQL, caching (Redis), and message queues (Celery)
- Familiarity with CI/CD pipelines (GitHub Actions, Jenkins)
- Excellent communication and problem-solving skills

Compensation:
- CTC: ₹12–18 LPA (based on experience)
- Health insurance, PF, and annual performance bonuses included

Apply at: careers@technova.io | www.technova.io/careers/swe-backend
No application fee. No advance payment. Strictly merit-based selection.
"""

# ══════════════════════════════════════════════════════════════
#  TEST B : Scam Job Description
# ══════════════════════════════════════════════════════════════
scam_job = """
URGENT HIRING!!! WORK FROM HOME — EARN Rs.50,000 MONTHLY!!!

We are hiring IMMEDIATELY for DATA ENTRY and TYPING WORK FROM HOME.
NO EXPERIENCE NEEDED. GUARANTEED INCOME every month.

Requirements:
- Anyone can apply! Students, housewives, retired persons welcome.
- Basic smartphone or laptop required.
- Must pay a ONE-TIME registration fee of Rs.999 to receive work kit.
- Processing fee of Rs.500 also applicable.

APPLY NOW!! LIMITED SEATS ONLY. Don't miss this ONCE IN A LIFETIME chance.
Earn money fast with UNLIMITED EARNINGS potential!!

Contact us on WHATSAPP: +91-XXXXXXXXXX
Or message on TELEGRAM: @scam_recruiter
Send CV to: fastjobs2024@gmail.com

Immediate joining required. Interview TODAY only.
Act now — only 5 slots remaining!!
"""


# ── Run Analysis ──
result_genuine = analyze_job(genuine_job)
result_scam    = analyze_job(scam_job)


# ── Display Results ──
print("=" * 65)
print("  TEST A : Genuine Job Description")
print("=" * 65)
print(json.dumps(result_genuine, indent=4))

print()
print("=" * 65)
print("  TEST B : Scam Job Description")
print("=" * 65)
print(json.dumps(result_scam, indent=4))

  TEST A : Genuine Job Description
{
    "risk_score": 34.1,
    "risk_level": "LOW",
    "trust_score": 65.9,
    "trust_level": "High Trust",
    "ml_prediction": 1,
    "ml_probability": 0.5525,
    "rule_score": 20,
    "fraud_reasons": [
        "Advance payment mentioned"
    ],
    "recommended_action": "Safe to Proceed"
}

  TEST B : Scam Job Description
{
    "risk_score": 70.68,
    "risk_level": "HIGH",
    "trust_score": 29.32,
    "trust_level": "Low Trust",
    "ml_prediction": 0,
    "ml_probability": 0.267,
    "rule_score": 100,
    "fraud_reasons": [
        "Registration fee mentioned",
        "Processing fee detected",
        "'Earn money fast' language detected",
        "Guaranteed income promise detected",
        "Typing-job scam pattern detected",
        "Telegram contact used (no official channel)",
        "WhatsApp communication detected",
        "Personal Gmail address used for recruitment",
        "High-pressure 'Apply Now' language",
        "Urgent 

In [16]:

#  STEP 15 (continued) : Summary Comparison Table

summary_df = pd.DataFrame([
    {
        "Input"             : "Genuine Job",
        "Risk Score"        : result_genuine["risk_score"],
        "Risk Level"        : result_genuine["risk_level"],
        "Trust Score"       : result_genuine["trust_score"],
        "Trust Level"       : result_genuine["trust_level"],
        "ML Probability"    : result_genuine["ml_probability"],
        "Rule Score"        : result_genuine["rule_score"],
        "Recommendation"    : result_genuine["recommended_action"],
        "Fraud Reasons"     : " | ".join(result_genuine["fraud_reasons"]),
    },
    {
        "Input"             : "Scam Job",
        "Risk Score"        : result_scam["risk_score"],
        "Risk Level"        : result_scam["risk_level"],
        "Trust Score"       : result_scam["trust_score"],
        "Trust Level"       : result_scam["trust_level"],
        "ML Probability"    : result_scam["ml_probability"],
        "Rule Score"        : result_scam["rule_score"],
        "Recommendation"    : result_scam["recommended_action"],
        "Fraud Reasons"     : " | ".join(result_scam["fraud_reasons"]),
    },
])

print("\n📊 Side-by-Side Comparison:")
print(summary_df.T.to_string())


📊 Side-by-Side Comparison:
                                        0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    1
Input                         Genuine Job                                                                                                                                                                                                                                                                                                                                                                                                                                    

---
## STEP 16 : Export Sample Results

The two test results are saved to `../data/explainable_jobs.csv` for:
- Project report appendix
- Frontend demo data
- Audit trail of engine outputs

In [17]:

#  STEP 16 : Export Sample Results

OUTPUT_CSV = "../data/explainable_jobs.csv"

export_df = pd.DataFrame([
    {
        "input_label"       : "Genuine Job Description",
        "risk_score"        : result_genuine["risk_score"],
        "risk_level"        : result_genuine["risk_level"],
        "trust_score"       : result_genuine["trust_score"],
        "trust_level"       : result_genuine["trust_level"],
        "ml_prediction"     : result_genuine["ml_prediction"],
        "ml_probability"    : result_genuine["ml_probability"],
        "rule_score"        : result_genuine["rule_score"],
        "fraud_reasons"     : " | ".join(result_genuine["fraud_reasons"]),
        "recommended_action": result_genuine["recommended_action"],
    },
    {
        "input_label"       : "Scam Job Description",
        "risk_score"        : result_scam["risk_score"],
        "risk_level"        : result_scam["risk_level"],
        "trust_score"       : result_scam["trust_score"],
        "trust_level"       : result_scam["trust_level"],
        "ml_prediction"     : result_scam["ml_prediction"],
        "ml_probability"    : result_scam["ml_probability"],
        "rule_score"        : result_scam["rule_score"],
        "fraud_reasons"     : " | ".join(result_scam["fraud_reasons"]),
        "recommended_action": result_scam["recommended_action"],
    },
])

export_df.to_csv(OUTPUT_CSV, index=False)

print(f" Results exported to : {OUTPUT_CSV}")
print(f"   Rows saved          : {len(export_df)}")
print(f"   Columns             : {list(export_df.columns)}")

✅ Results exported to : ../data/explainable_jobs.csv
   Rows saved          : 2
   Columns             : ['input_label', 'risk_score', 'risk_level', 'trust_score', 'trust_level', 'ml_prediction', 'ml_probability', 'rule_score', 'fraud_reasons', 'recommended_action']


---
## ✅ Engine Summary

| Step | Component | Status |
|------|-----------|--------|
| 1 | Library Imports | ✅ |
| 2 | Model + Feature Names Loaded | ✅ |
| 3 | Fraud Dictionaries Defined | ✅ |
| 4 | NLP Feature Generator | ✅ |
| 5 | Feature Engineering Generator | ✅ |
| 6 | Model Input Builder | ✅ |
| 7 | ML Prediction Engine | ✅ |
| 8 | Rule-Based Fraud Engine | ✅ |
| 9 | Hybrid Risk Engine | ✅ |
| 10 | Risk Classifier | ✅ |
| 11 | Trust Classifier | ✅ |
| 12 | Explainability Layer | ✅ |
| 13 | Recommendation Engine | ✅ |
| 14 | `analyze_job()` JSON Generator | ✅ |
| 15 | Frontend Simulation Tests | ✅ |
| 16 | CSV Export | ✅ |

---

### 🔗 Integration Points

The frontend should call `analyze_job(raw_text)` with text obtained from:

| Frontend Module | Text Source |
|---|---|
| Job Description Analyser | Direct user input |
| URL Analyser | BeautifulSoup / requests scrape output |
| Screenshot Analyser | Tesseract OCR output |
| Offer Letter Analyser | PyMuPDF / pdfplumber text extraction |

All four sources produce plain text → `analyze_job()` → JSON response.

---
*Notebook 5 — Production Detection Engine | Fake Internship & Job Scam Detection System*